In [4]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt

### u2net_next_inpainting ####
base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all"
all_information_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real_images_uid_prompt.csv", dtype={"uid": str})
pickscore_df = pd.read_csv(os.path.join(base_dir, "pickscore/pickscore_score.csv"), dtype={"uid": str})
anime_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/qwen3_anime/qwen3_anime.csv", dtype={"uid": str})

valid_real_image_uids = anime_df[anime_df['anime'] == 'no']['uid'].tolist()
valid_pickscore_uids = pickscore_df[pickscore_df['real_image_score'] - pickscore_df['fake_image_score'] > 0.02]['uid'].tolist()
valid_uids = list(set(valid_real_image_uids) & set(valid_pickscore_uids))

print(f"valid_uids: {len(valid_uids)}")
print(f"valid_real_image_uids: {len(valid_real_image_uids)} / {len(anime_df)}")
print(f"valid_pickscore_uids: {len(valid_pickscore_uids)} / {len(pickscore_df)}")

random.seed(42)
valid_uids = random.sample(valid_uids, 9984)

ext_list = [".png", ".PNG", ".jpg", ".jpeg", ".JPG", ".JPEG"]
win_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all/real"
lose_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all/fake"
total_information_list = []
for uid in valid_uids:
    for ext in ext_list:
        win_image_path = os.path.join(win_image_dir, f"{uid}{ext}")
        if os.path.exists(win_image_path):
            break
    if win_image_path is None:
        print(f"Warning: No valid image found for uid {uid}")
        continue
    
    for ext in ext_list:
        lose_image_path = os.path.join(lose_image_dir, f"{uid}{ext}")
        if os.path.exists(lose_image_path):
            break
    if lose_image_path is None:
        print(f"Warning: No valid image found for uid {uid}")
        continue
    
    prompt = all_information_df.loc[all_information_df['uid'] == uid, 'prompt'].values[0]
    total_information_list.append({
        "uid": uid,
        "prompt": prompt,
        "win_image_path": win_image_path,
        "lose_image_path": lose_image_path
    })
    
print(f"\nFinal valid samples: {len(total_information_list)}")
final_output_path = os.path.join(base_dir, "random_9984_images_no_anime_pickscore_0.02-hpdv3_all-uids.csv")
pd.DataFrame(total_information_list).to_csv(final_output_path, index=False)
print(f"Saved to: {final_output_path}")

valid_uids: 12157
valid_real_image_uids: 25096 / 42238
valid_pickscore_uids: 12157 / 25096

Final valid samples: 9984
Saved to: /data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all/random_9984_images_no_anime_pickscore_0.02-hpdv3_all-uids.csv
